# History of Artificial Intelligence: A Hands-on Notebook

**Track 01: AI and LLM Fundamentals · Module 1.1 · Beginner**

This notebook goes with the lesson *History of Artificial Intelligence: From Turing to AI Agents*. Instead of only reading about AI history, you will **rebuild five landmark systems in plain Python** and watch each one succeed or fail:

| # | Exhibit | Year | Paradigm | What you will see |
|---|---|---|---|---|
| 1 | A timeline you can query | 1943 → today | – | The eras, booms and winters |
| 2 | ELIZA chatbot | 1966 | Symbolic | Fluent replies with **zero understanding** |
| 3 | Perceptron | 1958 / 1969 | Connectionist | Learns AND and OR, **fails on XOR** |
| 4 | Expert system | 1980s | Symbolic | Explains its reasoning, but is **brittle** |
| 5 | Neural network + backpropagation | 1986 | Connectionist | **Solves XOR** |
| 6 | Bigram language model | 1990s–2000s | Statistical | Next-word prediction, **the idea behind LLMs** |

**How to use this notebook**

- Run the cells **from top to bottom** (`Shift + Enter` runs a cell and moves to the next one). Later cells use classes defined in earlier ones.
- Everything uses the **Python standard library only**. No API keys, no internet, no `pip install`.
- Cells marked **🧪 Try it yourself** are for experimenting. Change the values and re-run them.
- All randomness is **seeded**, so your numbers should match the lesson exactly.

> This is **demonstration code** written for clarity, not production code.

## 0. Setup

We only need modules from Python's standard library:

- `dataclasses`: quick, readable classes for data records.
- `enum`: named constants (our eras and paradigms).
- `re`: regular expressions (text patterns) for ELIZA and tokenising.
- `math` and `random`: for the neural network.
- `collections`: `Counter` and `defaultdict` for counting.

`from __future__ import annotations` lets us use modern type hints such as `list[str] | None` on older Python versions.

In [1]:
from __future__ import annotations

import math
import random
import re
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from enum import Enum

print("Python version:", sys.version.split()[0])

Python version: 3.14.5


---
## 1. Exhibit 1: A timeline you can query

AI history is not a straight line. It goes in **cycles**: an exciting idea → big promises → a hard limit → funding collapses (an **AI winter**) → a new idea revives the field.

We will model the history as data so we can ask questions such as *"what happened in the 1980s?"*.

### 1.1 Eras

An `Enum` is a fixed set of named values. Each era stores a **label**, a **start year** and an **end year**. `9999` means "still ongoing".

> The year ranges are **approximate teaching aids**. Historians draw the boundaries differently.

In [2]:
class Era(Enum):
    # Broad periods of AI history: (label, start year, end year)
    FOUNDATIONS = ("Foundations", 1943, 1955)
    BIRTH_AND_GOLDEN_YEARS = ("Birth and golden years", 1956, 1973)
    FIRST_WINTER = ("First AI winter", 1974, 1979)
    EXPERT_SYSTEMS_BOOM = ("Expert systems boom", 1980, 1986)
    SECOND_WINTER = ("Second AI winter", 1987, 1993)
    MACHINE_LEARNING_RISE = ("Rise of machine learning", 1994, 2011)
    DEEP_LEARNING = ("Deep learning revolution", 2012, 2016)
    TRANSFORMERS = ("Transformers and LLMs", 2017, 2021)
    GENERATIVE_AND_AGENTIC = ("Generative and agentic AI", 2022, 9999)

    def __init__(self, label: str, start: int, end: int) -> None:
        # Python passes the tuple's three values in here, so each member gets attributes.
        self.label = label
        self.start = start
        self.end = end

    def contains(self, year: int) -> bool:
        return self.start <= year <= self.end


print(Era.DEEP_LEARNING.label, "starts in", Era.DEEP_LEARNING.start)
print("Is 1975 in the first winter?", Era.FIRST_WINTER.contains(1975))

Deep learning revolution starts in 2012
Is 1975 in the first winter? True


### 1.2 Paradigms: the three schools of AI (plus agents)

| Paradigm | Core idea | Example |
|---|---|---|
| **Symbolic** | Humans write explicit rules and logic | ELIZA, expert systems, Deep Blue |
| **Connectionist** | Networks of simple "neurons" learn from data | Perceptron, deep learning, LLMs |
| **Statistical** | Learn probabilities and patterns from data | Spam filters, n-gram models |
| **Agentic** | Systems that perceive, reason and act in a loop | Shakey, ReAct, today's AI agents |

Inheriting from `str` makes each member behave like a normal string, which is handy for printing.

In [3]:
class Paradigm(str, Enum):
    SYMBOLIC = "symbolic"            # hand-written rules and logic
    CONNECTIONIST = "connectionist"  # neural networks
    STATISTICAL = "statistical"      # probability and classic ML
    AGENTIC = "agentic"              # perceive-reason-act systems
    INSTITUTIONAL = "institutional"  # funding, reports, events


print(Paradigm.CONNECTIONIST.value)

connectionist


### 1.3 A milestone record

`@dataclass(frozen=True)` writes `__init__` and `__repr__` for us. `frozen=True` makes each milestone **immutable** (read-only), because history should not be edited by accident.

The `era` is a **computed property**: it is worked out from the year each time, so it can never disagree with the year.

In [4]:
def era_for_year(year: int) -> Era:
    # Return the era containing `year`, or raise a clear error.
    for era in Era:
        if era.contains(year):
            return era
    raise ValueError(f"No era defined for year {year}; the timeline starts in 1943.")


@dataclass(frozen=True)
class Milestone:
    year: int
    title: str
    people: str
    paradigm: Paradigm
    why_it_matters: str

    @property
    def era(self) -> Era:
        return era_for_year(self.year)


example = Milestone(1950, "Turing's paper", "Alan Turing", Paradigm.SYMBOLIC, "Asked 'Can machines think?'")
print(example)
print("Era:", example.era.label)

Milestone(year=1950, title="Turing's paper", people='Alan Turing', paradigm=<Paradigm.SYMBOLIC: 'symbolic'>, why_it_matters="Asked 'Can machines think?'")
Era: Foundations


### 1.4 The data

Here are 35 milestones from 1943 to 2024. Read through them. Each `why_it_matters` field is a one-line history lesson. Notice how few entries fall inside the two **winters**.

In [5]:
MILESTONES: tuple[Milestone, ...] = (
    Milestone(1943, "Artificial neuron model", "Warren McCulloch, Walter Pitts",
              Paradigm.CONNECTIONIST, "First mathematical model of a neuron as a logic unit."),
    Milestone(1950, "Turing's 'Computing Machinery and Intelligence'", "Alan Turing",
              Paradigm.SYMBOLIC, "Asked 'Can machines think?' and proposed the imitation game."),
    Milestone(1956, "Dartmouth workshop names the field", "John McCarthy, Marvin Minsky, Claude Shannon, Nathaniel Rochester",
              Paradigm.INSTITUTIONAL, "The term 'artificial intelligence' becomes the name of a research field."),
    Milestone(1956, "Logic Theorist", "Allen Newell, Herbert Simon, Cliff Shaw",
              Paradigm.SYMBOLIC, "A program that proved theorems from Principia Mathematica."),
    Milestone(1958, "Perceptron", "Frank Rosenblatt",
              Paradigm.CONNECTIONIST, "A neuron-like model that learns its weights from examples."),
    Milestone(1958, "LISP programming language", "John McCarthy",
              Paradigm.SYMBOLIC, "Became the dominant language of AI research for decades."),
    Milestone(1959, "'Machine learning' coined via checkers program", "Arthur Samuel",
              Paradigm.STATISTICAL, "A program that improved at checkers by playing against itself."),
    Milestone(1966, "ELIZA chatbot", "Joseph Weizenbaum",
              Paradigm.SYMBOLIC, "Pattern-matching chat that people treated as if it understood them."),
    Milestone(1969, "'Perceptrons' book", "Marvin Minsky, Seymour Papert",
              Paradigm.CONNECTIONIST, "Showed single-layer perceptrons cannot learn XOR; neural research cooled."),
    Milestone(1972, "Shakey the robot completes its project", "SRI International (Nilsson and others)",
              Paradigm.AGENTIC, "First mobile robot to perceive, plan (STRIPS, A*) and act: an early agent."),
    Milestone(1973, "Lighthill report", "James Lighthill",
              Paradigm.INSTITUTIONAL, "Critical UK report that led to major funding cuts."),
    Milestone(1976, "MYCIN expert system", "Edward Shortliffe and the Stanford team",
              Paradigm.SYMBOLIC, "Rule-based diagnosis of blood infections with explanations."),
    Milestone(1980, "XCON (R1) in production at DEC", "John McDermott, Digital Equipment Corporation",
              Paradigm.SYMBOLIC, "Commercial expert system that configured computer orders."),
    Milestone(1986, "Backpropagation popularised", "David Rumelhart, Geoffrey Hinton, Ronald Williams",
              Paradigm.CONNECTIONIST, "Made multi-layer neural networks trainable; XOR is solved."),
    Milestone(1987, "Lisp machine market collapses", "Industry-wide",
              Paradigm.INSTITUTIONAL, "Expert-system hype fades; the second AI winter begins."),
    Milestone(1989, "Convolutional network reads handwritten digits", "Yann LeCun and colleagues",
              Paradigm.CONNECTIONIST, "Early practical neural network for postal zip codes."),
    Milestone(1995, "'AI: A Modern Approach' frames AI as agents", "Stuart Russell, Peter Norvig",
              Paradigm.AGENTIC, "Textbook that defines AI as building rational agents."),
    Milestone(1997, "Deep Blue defeats Garry Kasparov", "IBM",
              Paradigm.SYMBOLIC, "Search plus hand-tuned evaluation beats the chess world champion."),
    Milestone(1997, "LSTM networks", "Sepp Hochreiter, Juergen Schmidhuber",
              Paradigm.CONNECTIONIST, "Recurrent networks that remember over long sequences."),
    Milestone(2006, "Deep belief networks revive 'deep learning'", "Geoffrey Hinton and colleagues",
              Paradigm.CONNECTIONIST, "Showed deep networks could be trained layer by layer."),
    Milestone(2009, "ImageNet dataset", "Fei-Fei Li and colleagues",
              Paradigm.STATISTICAL, "Millions of labelled images: the fuel for deep learning."),
    Milestone(2011, "IBM Watson wins Jeopardy!", "IBM",
              Paradigm.STATISTICAL, "Question answering over large text collections."),
    Milestone(2012, "AlexNet wins ImageNet challenge", "Alex Krizhevsky, Ilya Sutskever, Geoffrey Hinton",
              Paradigm.CONNECTIONIST, "GPU-trained deep network cuts image error dramatically."),
    Milestone(2013, "word2vec embeddings", "Tomas Mikolov and colleagues (Google)",
              Paradigm.CONNECTIONIST, "Words become vectors whose geometry captures meaning."),
    Milestone(2014, "Generative adversarial networks", "Ian Goodfellow and colleagues",
              Paradigm.CONNECTIONIST, "Two networks compete to generate realistic data."),
    Milestone(2016, "AlphaGo defeats Lee Sedol", "DeepMind",
              Paradigm.CONNECTIONIST, "Deep learning plus reinforcement learning masters Go."),
    Milestone(2017, "Transformer architecture", "Ashish Vaswani and colleagues (Google)",
              Paradigm.CONNECTIONIST, "'Attention Is All You Need': the architecture behind modern LLMs."),
    Milestone(2018, "BERT and GPT-1", "Google; OpenAI",
              Paradigm.CONNECTIONIST, "Pre-train once on huge text, then adapt to many tasks."),
    Milestone(2020, "GPT-3", "OpenAI",
              Paradigm.CONNECTIONIST, "175-billion-parameter model shows few-shot learning from prompts."),
    Milestone(2020, "Retrieval-augmented generation (RAG)", "Patrick Lewis and colleagues (Facebook AI)",
              Paradigm.AGENTIC, "Combines a language model with document retrieval."),
    Milestone(2022, "ReAct: reasoning plus acting", "Shunyu Yao and colleagues",
              Paradigm.AGENTIC, "LLMs interleave thoughts with tool calls: the modern agent loop."),
    Milestone(2022, "ChatGPT released", "OpenAI",
              Paradigm.CONNECTIONIST, "Conversational LLM reaches the general public."),
    Milestone(2023, "Function calling / tool use in LLM APIs", "Multiple vendors",
              Paradigm.AGENTIC, "Models return structured tool calls that programs can execute."),
    Milestone(2024, "Model Context Protocol (MCP)", "Anthropic",
              Paradigm.AGENTIC, "Open standard for connecting AI applications to tools and data."),
    Milestone(2024, "Nobel Prizes recognise AI research", "John Hopfield, Geoffrey Hinton; Demis Hassabis, John Jumper",
              Paradigm.INSTITUTIONAL, "Physics prize for neural networks; Chemistry prize for AlphaFold."),
)

print(f"Loaded {len(MILESTONES)} milestones from {MILESTONES[0].year} to {MILESTONES[-1].year}")

Loaded 35 milestones from 1943 to 2024


### 1.5 Asking questions of the timeline

Three small query functions:

- `milestones_in_era(era)`: everything in one era.
- `search(keyword)`: a case-insensitive search across title, people and explanation.
- `count_by_paradigm()`: how often each school of thought produced a milestone.

Note the **input validation**: `search("")` raises a `ValueError` instead of silently returning everything.

In [6]:
def milestones_in_era(era: Era) -> list[Milestone]:
    return sorted((m for m in MILESTONES if m.era is era), key=lambda m: m.year)


def search(keyword: str) -> list[Milestone]:
    needle = keyword.strip().lower()
    if not needle:
        raise ValueError("keyword must not be empty")
    return [m for m in MILESTONES
            if needle in m.title.lower() or needle in m.people.lower() or needle in m.why_it_matters.lower()]


def count_by_paradigm() -> dict[Paradigm, int]:
    counts: dict[Paradigm, int] = defaultdict(int)  # missing keys start at 0
    for m in MILESTONES:
        counts[m.paradigm] += 1
    return dict(counts)

In [7]:
print("Milestones per era")
print("-" * 52)
for era in Era:
    end = "today" if era.end == 9999 else str(era.end)
    count = len(milestones_in_era(era))
    print(f"{era.start}-{end:<6} {era.label:<27} {'#' * count} {count}")

Milestones per era
----------------------------------------------------
1943-1955   Foundations                 ## 2
1956-1973   Birth and golden years      ######### 9
1974-1979   First AI winter             # 1
1980-1986   Expert systems boom         ## 2
1987-1993   Second AI winter            ## 2
1994-2011   Rise of machine learning    ###### 6
2012-2016   Deep learning revolution    #### 4
2017-2021   Transformers and LLMs       #### 4
2022-today  Generative and agentic AI   ##### 5


**What to notice:** the `#` bars are a mini bar chart. The two **winters** (1974–1979 and 1987–1993) are short and nearly empty. Progress depends on funding and belief, not only on ideas.

In [8]:
print("Milestones per paradigm:")
for paradigm, count in sorted(count_by_paradigm().items(), key=lambda kv: -kv[1]):
    print(f"  {paradigm.value:<14} {count}")

print("\nWhat happened in the expert systems boom?")
for m in milestones_in_era(Era.EXPERT_SYSTEMS_BOOM):
    print(f"  {m.year}  {m.title}: {m.why_it_matters}")

Milestones per paradigm:
  connectionist  15
  symbolic       7
  agentic        6
  institutional  4
  statistical    3

What happened in the expert systems boom?
  1980  XCON (R1) in production at DEC: Commercial expert system that configured computer orders.
  1986  Backpropagation popularised: Made multi-layer neural networks trainable; XOR is solved.


### 🧪 Try it yourself

Change the keyword below and re-run the cell. Some ideas: `"agent"`, `"Hinton"`, `"chess"`, `"language"`.

In [9]:
keyword = "neural"
hits = search(keyword)
print(f"{len(hits)} milestone(s) match '{keyword}':")
for m in hits:
    print(f"  {m.year}  {m.title}  [{m.era.label}]")

4 milestone(s) match 'neural':
  1969  'Perceptrons' book  [Birth and golden years]
  1986  Backpropagation popularised  [Expert systems boom]
  1989  Convolutional network reads handwritten digits  [Second AI winter]
  2024  Nobel Prizes recognise AI research  [Generative and agentic AI]


---
## 2. Exhibit 2: ELIZA (1966), conversation by pattern matching

**History:** Joseph Weizenbaum at MIT wrote ELIZA, whose most famous script imitated a psychotherapist. Users confided in it and some were convinced it understood them, which alarmed Weizenbaum. Today this is called the **ELIZA effect**: people attribute understanding to anything that produces fluent language.

**How it works:**

1. Check the user's sentence against a list of **patterns** (regular expressions), in order.
2. **Capture** part of the sentence, for example everything after "I am".
3. **Reflect** pronouns: "my" → "your", "I" → "you".
4. Insert the reflected text into a **template** reply.

There is **no understanding and no learning**. The "intelligence" is entirely in the hand-written rules, which makes this **symbolic AI**.

### 2.1 Pronoun reflection

In [10]:
REFLECTIONS: dict[str, str] = {
    "i": "you", "me": "you", "my": "your", "am": "are", "i'm": "you are",
    "you": "I", "your": "my", "yours": "mine", "myself": "yourself",
}


def reflect(fragment: str) -> str:
    # Swap first and second person, word by word.
    return " ".join(REFLECTIONS.get(word, word) for word in fragment.lower().split())


print(reflect("I am worried about my exams"))

you are worried about your exams


### 2.2 Pattern rules

Each `PatternRule` pairs a compiled regex with one or more reply templates. `{0}` in a template is replaced with whatever `(.+)` captured.

Instead of picking a reply at random, `next_response()` **rotates** through them. This makes the output **deterministic** (the same every time), which is much easier to test.

The **last rule `(.*)` matches anything**. It is a catch-all that guarantees a reply, and it is why ELIZA often says something vague.

In [11]:
@dataclass
class PatternRule:
    pattern: re.Pattern[str]
    responses: tuple[str, ...]
    _next: int = field(default=0, repr=False)

    def next_response(self) -> str:
        response = self.responses[self._next % len(self.responses)]
        self._next += 1
        return response


def _rule(regex: str, *responses: str) -> PatternRule:
    return PatternRule(re.compile(regex, re.IGNORECASE), responses)


def default_rules() -> list[PatternRule]:
    # Checked top to bottom. The last one matches anything.
    return [
        _rule(r"\bi need (.+)", "Why do you need {0}?", "Would getting {0} really help you?"),
        _rule(r"\bi am (.+)", "How long have you been {0}?", "Why do you say you are {0}?"),
        _rule(r"\bi feel (.+)", "Tell me more about feeling {0}.", "Do you often feel {0}?"),
        _rule(r"\bbecause (.+)", "Is that the real reason?", "What else could explain {0}?"),
        _rule(r"\b(?:mother|father|family)\b", "Tell me more about your family."),
        _rule(r"\bcomputer|machine|ai\b", "Do machines worry you?", "What do you think about machines?"),
        _rule(r"(.*)", "Please tell me more.", "I see. Go on.", "How does that make you feel?"),
    ]

### 2.3 The chatbot

`respond()` cleans the input, finds the **first** matching rule, reflects the captured text and fills in the template.

In [12]:
class Eliza:
    def __init__(self, rules: list[PatternRule] | None = None) -> None:
        self.rules = rules if rules is not None else default_rules()

    def respond(self, text: str) -> str:
        cleaned = text.strip().rstrip(".!?")
        if not cleaned:
            raise ValueError("text must not be empty")
        for rule in self.rules:
            match = rule.pattern.search(cleaned)
            if match:
                captured = [reflect(group) for group in match.groups()]
                return rule.next_response().format(*captured)
        return "Please go on."


bot = Eliza()
for line in ("I am worried about my exams", "I need a holiday", "Do you understand me?"):
    print(f"User : {line}")
    print(f"ELIZA: {bot.respond(line)}\n")

User : I am worried about my exams
ELIZA: How long have you been worried about your exams?

User : I need a holiday
ELIZA: Why do you need a holiday?

User : Do you understand me?
ELIZA: Please tell me more.


**What to notice:**

- "my exams" became "**your** exams". That is the reflection trick, and it feels personal.
- "Do you understand me?" got a generic reply, because no rule covers it. ELIZA cannot answer the most important question of all.

**Link to today:** LLMs are vastly more capable, but the **ELIZA effect** is stronger than ever. When you build agents, show sources, admit uncertainty and never let fluent text stand in for verified facts.

### 🧪 Try it yourself

Type your own messages. Can you find a sentence that makes ELIZA reply with something silly?

In [13]:
my_bot = Eliza()
for line in ["I feel tired because I stayed up coding", "My father loves computers", "I am an AI engineer"]:
    print(f"User : {line}\nELIZA: {my_bot.respond(line)}\n")

User : I feel tired because I stayed up coding
ELIZA: Tell me more about feeling tired because you stayed up coding.

User : My father loves computers
ELIZA: Tell me more about your family.

User : I am an AI engineer
ELIZA: How long have you been an ai engineer?


---
## 3. Exhibit 3: The perceptron (1958) and the XOR problem (1969)

**History:** Frank Rosenblatt's **perceptron** was a learning machine inspired by brain neurons, and it caused huge excitement. In 1969 Marvin Minsky and Seymour Papert's book *Perceptrons* showed rigorously that a single perceptron **cannot learn XOR**. Interest in neural networks cooled for more than a decade.

**How a perceptron works:**

```
total  = w1*x1 + w2*x2 + bias
output = 1 if total > 0 else 0          (a "step" function)
```

**How it learns (the perceptron learning rule):** after each example, compute `error = target - output`, which is `+1`, `-1` or `0`. Then nudge every weight:

```
w_i  += learning_rate * error * x_i
bias += learning_rate * error
```

### 3.1 The datasets

Each sample is `((x1, x2), target)`.

| x1 | x2 | AND | OR | XOR |
|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 |
| 0 | 1 | 0 | 1 | **1** |
| 1 | 0 | 0 | 1 | **1** |
| 1 | 1 | 1 | 1 | **0** |

XOR ("exclusive or") is 1 when **exactly one** input is 1.

In [14]:
Sample = tuple[tuple[float, float], int]

AND_DATA: list[Sample] = [((0, 0), 0), ((0, 1), 0), ((1, 0), 0), ((1, 1), 1)]
OR_DATA: list[Sample] = [((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 1), 1)]
XOR_DATA: list[Sample] = [((0, 0), 0), ((0, 1), 1), ((1, 0), 1), ((1, 1), 0)]

### 3.2 The Perceptron class

- `predict()`: the weighted sum plus a step function.
- `fit()`: loops over the data for several **epochs** (one epoch = one pass through all samples) and stops early once an epoch has **zero mistakes**. It returns the number of mistakes in each epoch so we can see the learning progress.

In [15]:
class Perceptron:
    def __init__(self, n_inputs: int = 2, learning_rate: float = 0.1) -> None:
        if n_inputs < 1:
            raise ValueError("n_inputs must be at least 1")
        if learning_rate <= 0:
            raise ValueError("learning_rate must be positive")
        self.weights: list[float] = [0.0] * n_inputs
        self.bias: float = 0.0
        self.learning_rate = learning_rate

    def predict(self, inputs: tuple[float, ...]) -> int:
        if len(inputs) != len(self.weights):
            raise ValueError(f"expected {len(self.weights)} inputs, got {len(inputs)}")
        total = sum(w * x for w, x in zip(self.weights, inputs)) + self.bias
        return 1 if total > 0 else 0

    def fit(self, data: list[Sample], epochs: int = 20) -> list[int]:
        mistakes_per_epoch: list[int] = []
        for _ in range(epochs):
            mistakes = 0
            for inputs, target in data:
                error = target - self.predict(inputs)   # -1, 0 or +1
                if error != 0:
                    mistakes += 1
                    for i, x in enumerate(inputs):
                        self.weights[i] += self.learning_rate * error * x
                    self.bias += self.learning_rate * error
            mistakes_per_epoch.append(mistakes)
            if mistakes == 0:
                break
        return mistakes_per_epoch

    def accuracy(self, data: list[Sample]) -> float:
        return sum(1 for inputs, target in data if self.predict(inputs) == target) / len(data)

### 3.3 Watch it learn AND, step by step

Let's look inside training: the mistakes in each epoch, and the final weights.

In [16]:
p = Perceptron()
history = p.fit(AND_DATA, epochs=20)
print("Mistakes per epoch:", history)
print("Learned weights:", [round(w, 2) for w in p.weights], " bias:", round(p.bias, 2))
for inputs, target in AND_DATA:
    print(f"  {inputs} -> predicted {p.predict(inputs)}  (target {target})")

Mistakes per epoch: [1, 3, 3, 2, 1, 0]
Learned weights: [0.2, 0.1]  bias: -0.2
  (0, 0) -> predicted 0  (target 0)
  (0, 1) -> predicted 0  (target 0)
  (1, 0) -> predicted 0  (target 0)
  (1, 1) -> predicted 1  (target 1)


The learned weights describe a **straight line**, `w1*x1 + w2*x2 + bias = 0`, with `(1, 1)` on one side and the other three points on the other. Training means **moving that line** until it separates the classes.

### 3.4 AND vs OR vs XOR

In [17]:
for name, data in (("AND", AND_DATA), ("OR", OR_DATA), ("XOR", XOR_DATA)):
    p = Perceptron()
    history = p.fit(data, epochs=20)
    print(f"{name:<3} accuracy {p.accuracy(data):>4.0%} after {len(history):>2} epochs "
          f"(mistakes in last epoch: {history[-1]})")

AND accuracy 100% after  6 epochs (mistakes in last epoch: 0)
OR  accuracy 100% after  4 epochs (mistakes in last epoch: 0)
XOR accuracy  50% after 20 epochs (mistakes in last epoch: 4)


### 3.5 Why XOR fails

Draw the four XOR points:

```
 x2
 1 |  (0,1)=1      (1,1)=0
 0 |  (0,0)=0      (1,0)=1
   +---------------------- x1
        0             1
```

The 1s sit on one **diagonal** and the 0s on the other. **No single straight line** can put both 1s on one side and both 0s on the other. XOR is **not linearly separable**, so a perceptron, which can only draw one line, can never learn it, however long it trains.

### 🧪 Try it yourself

Train for more epochs or change the learning rate. Does XOR ever reach 100%?

In [18]:
p = Perceptron(learning_rate=0.1)
history = p.fit(XOR_DATA, epochs=500)
print(f"Epochs run: {len(history)}  mistakes in last epoch: {history[-1]}  accuracy: {p.accuracy(XOR_DATA):.0%}")

Epochs run: 500  mistakes in last epoch: 4  accuracy: 50%


---
## 4. Exhibit 4: An expert system (1970s–1980s)

**History:** after the first AI winter, researchers aimed lower: instead of general intelligence, they captured **one expert's narrow knowledge** as IF-THEN rules. **MYCIN** (Stanford) diagnosed blood infections. **XCON** at Digital Equipment Corporation configured computer orders in production. This was AI's first big commercial success, and it collapsed in the late 1980s.

**The three classic parts:**

| Part | Role | In our code |
|---|---|---|
| **Knowledge base** | Rules written by experts | `HELPDESK_RULES` |
| **Inference engine** | Applies rules to the facts | `ExpertSystem.infer()` |
| **Explanation facility** | Answers "why do you think that?" | `InferenceResult.explain()` |

### 4.1 Rules and the knowledge base

A rule says: *if all of these conditions are known facts, then add this conclusion.* Conditions are stored as a `frozenset`, so "are all conditions known?" becomes a simple subset test: `conditions <= facts`.

Notice the **chain**: R1 concludes `power_supply_suspect`, and that fact is a condition of R2 and R3.

In [19]:
@dataclass(frozen=True)
class Rule:
    name: str
    conditions: frozenset[str]
    conclusion: str


def rule(name: str, conditions: set[str], conclusion: str) -> Rule:
    return Rule(name, frozenset(conditions), conclusion)


HELPDESK_RULES: tuple[Rule, ...] = (
    rule("R1", {"no_power_light", "cable_connected"}, "power_supply_suspect"),
    rule("R2", {"power_supply_suspect", "under_warranty"}, "action:request_replacement"),
    rule("R3", {"power_supply_suspect", "out_of_warranty"}, "action:quote_repair"),
    rule("R4", {"wifi_icon_crossed", "other_devices_online"}, "laptop_network_fault"),
    rule("R5", {"laptop_network_fault"}, "action:restart_network_adapter"),
    rule("R6", {"slow_boot", "disk_over_90_percent"}, "storage_pressure"),
    rule("R7", {"storage_pressure"}, "action:clean_up_disk"),
)

for r in HELPDESK_RULES:
    print(f"{r.name}: IF {' AND '.join(sorted(r.conditions))} THEN {r.conclusion}")

R1: IF cable_connected AND no_power_light THEN power_supply_suspect
R2: IF power_supply_suspect AND under_warranty THEN action:request_replacement
R3: IF out_of_warranty AND power_supply_suspect THEN action:quote_repair
R4: IF other_devices_online AND wifi_icon_crossed THEN laptop_network_fault
R5: IF laptop_network_fault THEN action:restart_network_adapter
R6: IF disk_over_90_percent AND slow_boot THEN storage_pressure
R7: IF storage_pressure THEN action:clean_up_disk


### 4.2 The inference result and its explanation

`InferenceResult` stores:

- `facts`: everything known at the end.
- `trace`: the rule names, in the order they fired.
- `explanations`: for each derived fact, the rule that produced it.

`explain()` is **recursive**. To explain a fact, it names the rule that produced it, then explains each of that rule's conditions, until it reaches facts the user supplied.

In [20]:
@dataclass
class InferenceResult:
    facts: set[str]
    trace: list[str] = field(default_factory=list)
    explanations: dict[str, Rule] = field(default_factory=dict)

    def explain(self, fact: str, depth: int = 0) -> list[str]:
        indent = "  " * depth
        rule_used = self.explanations.get(fact)
        if rule_used is None:
            return [f"{indent}{fact} (given by the user)"]
        lines = [f"{indent}{fact} because rule {rule_used.name} needs {sorted(rule_used.conditions)}"]
        for condition in sorted(rule_used.conditions):
            lines.extend(self.explain(condition, depth + 1))
        return lines

### 4.3 The inference engine (forward chaining)

**Forward chaining**, in plain English:

1. Start with the user's facts.
2. Go through every rule. If all its conditions are known and its conclusion is new, **fire** it: add the conclusion as a new fact.
3. If anything new was added, go through the rules again.
4. Stop when a full pass adds nothing. This is called a **fixed point**.

In [21]:
class ExpertSystem:
    def __init__(self, rules: tuple[Rule, ...] = HELPDESK_RULES) -> None:
        self.rules = rules

    def infer(self, initial_facts: set[str]) -> InferenceResult:
        if not initial_facts:
            raise ValueError("at least one fact is required")
        result = InferenceResult(facts=set(initial_facts))
        changed = True
        while changed:
            changed = False
            for r in self.rules:
                if r.conclusion not in result.facts and r.conditions <= result.facts:
                    result.facts.add(r.conclusion)
                    result.trace.append(r.name)
                    result.explanations[r.conclusion] = r
                    changed = True
        return result

    @staticmethod
    def actions(result: InferenceResult) -> list[str]:
        return sorted(f.removeprefix("action:") for f in result.facts if f.startswith("action:"))

### 4.4 Run a diagnosis

In [22]:
system = ExpertSystem()
facts = {"no_power_light", "cable_connected", "under_warranty"}
result = system.infer(facts)

print("Facts given :", sorted(facts))
print("Rules fired :", result.trace)
print("Recommended :", ExpertSystem.actions(result))
print("\nWhy?")
for line in result.explain("action:request_replacement"):
    print(" ", line)

Facts given : ['cable_connected', 'no_power_light', 'under_warranty']
Rules fired : ['R1', 'R2']
Recommended : ['request_replacement']

Why?
  action:request_replacement because rule R2 needs ['power_supply_suspect', 'under_warranty']
    power_supply_suspect because rule R1 needs ['cable_connected', 'no_power_light']
      cable_connected (given by the user)
      no_power_light (given by the user)
    under_warranty (given by the user)


**Strength: explainability.** Every conclusion can be traced back to rules and facts. That is why doctors were willing to trust MYCIN, and why regulated industries still use rule engines.

### 4.5 The weakness: brittleness

What happens with a symptom the experts never wrote a rule for?

In [23]:
unknown = system.infer({"screen_flickers"})
print("Recommended:", ExpertSystem.actions(unknown) or "nothing: no rule applies")

Recommended: nothing: no rule applies


**Nothing.** The system does not degrade gracefully. It simply has no answer. Multiply this by thousands of rules that must be written, checked and kept up to date by hand (the **knowledge acquisition bottleneck**), and you get the **second AI winter**.

**Link to today:** an agent whose behaviour lives in a long list of "if the user says X, do Y" prompt instructions can fail in exactly the same way.

### 🧪 Try it yourself

1. Change `facts` to `{"wifi_icon_crossed", "other_devices_online"}` and run the diagnosis.
2. Add a rule `R8` for `screen_flickers` and run it again.

In [24]:
my_rules = HELPDESK_RULES + (rule("R8", {"screen_flickers"}, "action:update_display_driver"),)
my_system = ExpertSystem(my_rules)
print(ExpertSystem.actions(my_system.infer({"screen_flickers"})))
print(ExpertSystem.actions(my_system.infer({"wifi_icon_crossed", "other_devices_online"})))

['update_display_driver']
['restart_network_adapter']


---
## 5. Exhibit 5: Backpropagation (1986) solves XOR

**History:** in 1986 David Rumelhart, Geoffrey Hinton and Ronald Williams published *Learning representations by back-propagating errors* in *Nature*. It showed how to train networks with **hidden layers**. This brought neural networks back, and the same principle trains every modern deep network, including LLMs.

**Two changes compared with the perceptron:**

1. **A hidden layer.** Hidden neurons create intermediate features, so the network can draw a **bent** boundary instead of one straight line.
2. **A smooth activation function (sigmoid).** The step function is flat almost everywhere, so it cannot tell us which way to change a weight. The sigmoid has a slope everywhere, so we can compute **gradients**.

```
inputs (2)  -->  hidden layer (4 neurons)  -->  output (1 neuron)
```

### 5.1 The sigmoid function

In [25]:
def sigmoid(z: float) -> float:
    return 1.0 / (1.0 + math.exp(-z))


for z in (-6, -2, 0, 2, 6):
    print(f"sigmoid({z:>2}) = {sigmoid(z):.3f}")

sigmoid(-6) = 0.002
sigmoid(-2) = 0.119
sigmoid( 0) = 0.500
sigmoid( 2) = 0.881
sigmoid( 6) = 0.998


It squashes any number into the range 0 to 1 smoothly. Big negative numbers go to about 0, big positive numbers to about 1, and 0 gives exactly 0.5.

Its slope is `s * (1 - s)`, where `s = sigmoid(z)`. We will use this in the backward pass.

### 5.2 The network

**Forward pass:** inputs → hidden activations → output probability.

**Backward pass (backpropagation):**

- `d_out = (out - target) * out * (1 - out)`: the error at the output, times the sigmoid's slope.
- `d_hidden[j] = d_out * w_out[j] * h * (1 - h)`: each hidden neuron's **share of the blame**, proportional to how strongly it connects to the output. This is the **chain rule** carrying the error backwards.
- Every weight moves a little **downhill**: `weight -= learning_rate * delta * input`. This is **gradient descent**.

`random.Random(seed)` gives random but **reproducible** starting weights. Random starting values matter because if all hidden neurons started equal, they would all learn the same thing.

In [26]:
class TinyMLP:
    def __init__(self, n_hidden: int = 4, seed: int = 7) -> None:
        if n_hidden < 2:
            raise ValueError("XOR needs at least 2 hidden neurons")
        rng = random.Random(seed)
        self.w_hidden = [[rng.uniform(-1, 1) for _ in range(2)] for _ in range(n_hidden)]
        self.b_hidden = [rng.uniform(-1, 1) for _ in range(n_hidden)]
        self.w_out = [rng.uniform(-1, 1) for _ in range(n_hidden)]
        self.b_out = rng.uniform(-1, 1)

    def _forward(self, x: tuple[float, float]) -> tuple[list[float], float]:
        hidden = [sigmoid(w[0] * x[0] + w[1] * x[1] + b) for w, b in zip(self.w_hidden, self.b_hidden)]
        output = sigmoid(sum(w * h for w, h in zip(self.w_out, hidden)) + self.b_out)
        return hidden, output

    def predict_proba(self, x: tuple[float, float]) -> float:
        return self._forward(x)[1]

    def predict(self, x: tuple[float, float]) -> int:
        return 1 if self.predict_proba(x) >= 0.5 else 0

    def fit(self, data: list[Sample], epochs: int = 5000, learning_rate: float = 0.5) -> list[float]:
        if not data:
            raise ValueError("data must not be empty")
        losses: list[float] = []
        for _ in range(epochs):
            total = 0.0
            for x, target in data:
                hidden, out = self._forward(x)
                total += (out - target) ** 2

                # Backward pass: chain rule from the output back to the inputs.
                d_out = (out - target) * out * (1 - out)
                d_hidden = [d_out * self.w_out[j] * h * (1 - h) for j, h in enumerate(hidden)]

                for j, h in enumerate(hidden):
                    self.w_out[j] -= learning_rate * d_out * h
                self.b_out -= learning_rate * d_out
                for j, d in enumerate(d_hidden):
                    self.w_hidden[j][0] -= learning_rate * d * x[0]
                    self.w_hidden[j][1] -= learning_rate * d * x[1]
                    self.b_hidden[j] -= learning_rate * d
            losses.append(total / len(data))
        return losses

    def accuracy(self, data: list[Sample]) -> float:
        return sum(1 for x, t in data if self.predict(x) == t) / len(data)

### 5.3 Before training

With random weights, the outputs are close to guesses.

In [27]:
net = TinyMLP(n_hidden=4, seed=7)
for x, target in XOR_DATA:
    print(f"input={x}  target={target}  output={net.predict_proba(x):.3f}")
print(f"Accuracy before training: {net.accuracy(XOR_DATA):.0%}")

input=(0, 0)  target=0  output=0.531
input=(0, 1)  target=1  output=0.513
input=(1, 0)  target=1  output=0.564
input=(1, 1)  target=0  output=0.544
Accuracy before training: 50%


### 5.4 Train and watch the loss fall

The **loss** (mean squared error) measures how wrong the network is. We print it every 500 epochs as a text bar chart: the bars should shrink.

In [28]:
losses = net.fit(XOR_DATA, epochs=5000, learning_rate=0.5)

for epoch in range(0, len(losses), 500):
    bar = "#" * int(losses[epoch] * 150)
    print(f"epoch {epoch + 1:>4}  loss {losses[epoch]:.4f}  {bar}")
print(f"epoch {len(losses):>4}  loss {losses[-1]:.4f}")

epoch    1  loss 0.2625  #######################################
epoch  501  loss 0.1925  ############################
epoch 1001  loss 0.0274  ####
epoch 1501  loss 0.0064  
epoch 2001  loss 0.0032  
epoch 2501  loss 0.0021  
epoch 3001  loss 0.0015  
epoch 3501  loss 0.0012  
epoch 4001  loss 0.0010  
epoch 4501  loss 0.0008  
epoch 5000  loss 0.0007


In [29]:
for x, target in XOR_DATA:
    print(f"input={x}  target={target}  output={net.predict_proba(x):.3f}  -> {net.predict(x)}")
print(f"XOR accuracy after training: {net.accuracy(XOR_DATA):.0%}")

input=(0, 0)  target=0  output=0.014  -> 0
input=(0, 1)  target=1  output=0.968  -> 1
input=(1, 0)  target=1  output=0.976  -> 1
input=(1, 1)  target=0  output=0.034  -> 0
XOR accuracy after training: 100%


**XOR is solved.** The problem that froze neural network research in 1969 falls to a hidden layer plus backpropagation.

Modern deep learning uses exactly this recipe (forward pass, loss, backward pass, gradient descent) with millions or billions of weights, faster maths libraries (NumPy, PyTorch) and GPUs. **AlexNet (2012)** showed that this recipe, at scale, beats everything else, and that started today's AI boom.

### 🧪 Try it yourself

Neural network training depends on the starting weights. Train with 20 different seeds and count how many reach 100%. Then try `n_hidden=2` and `n_hidden=8`. (This cell takes a few seconds.)

In [30]:
for n_hidden in (2, 4, 8):
    solved = 0
    for seed in range(20):
        candidate = TinyMLP(n_hidden=n_hidden, seed=seed)
        candidate.fit(XOR_DATA, epochs=2000, learning_rate=0.5)
        solved += candidate.accuracy(XOR_DATA) == 1.0
    print(f"n_hidden={n_hidden}: solved XOR with {solved}/20 seeds (2000 epochs)")

n_hidden=2: solved XOR with 12/20 seeds (2000 epochs)
n_hidden=4: solved XOR with 20/20 seeds (2000 epochs)
n_hidden=8: solved XOR with 20/20 seeds (2000 epochs)


---
## 6. Exhibit 6: A bigram language model, the idea behind LLMs

**History:** from the 1990s, **statistical** methods took over language technology. Speech recognition and machine translation used **n-gram models**, which predict the next word from the previous *n − 1* words by counting.

A **bigram** model looks at just **one** previous word:

```
P(next = "reads" | current = "agent")  =  count("agent reads") / count("agent")
```

A modern **LLM** does the same job, **predicting the next token**, but uses a Transformer neural network (2017) that looks at thousands of previous tokens, with billions of learned weights instead of a count table.

### 6.1 Tokenising and the training corpus

**Tokenising** splits text into units. Here we use lower-case words and full stops. LLMs split text into sub-word pieces called **tokens**.

`<s>` and `</s>` are special tokens that mark the start and end of a sentence. LLMs use similar special tokens.

In [31]:
START, END = "<s>", "</s>"

AGENT_CORPUS = '''
the agent reads the task .
the agent plans the next step .
the agent calls a tool .
the tool returns a result .
the agent reads the result .
the agent writes the answer .
'''


def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z']+|[.]", text.lower())


print(tokenize("The agent calls a tool."))

['the', 'agent', 'calls', 'a', 'tool', '.']


### 6.2 The model

- `train()`: for every sentence, walk through adjacent word pairs with `zip(tokens, tokens[1:])` and count them.
- `next_word_probabilities()`: turn counts into probabilities. An unknown word raises `KeyError`. This is the classic **out-of-vocabulary** problem, which sub-word tokens later solved.
- `generate()`: start at `<s>` and repeatedly **sample** the next word in proportion to its probability. This sampling step is what the `temperature` setting controls in LLM APIs.

In [32]:
class BigramModel:
    def __init__(self) -> None:
        self.counts: dict[str, Counter[str]] = defaultdict(Counter)

    def train(self, corpus: str) -> None:
        sentences = [line for line in corpus.strip().splitlines() if line.strip()]
        if not sentences:
            raise ValueError("corpus must contain at least one sentence")
        for sentence in sentences:
            tokens = [START, *tokenize(sentence), END]
            for current, nxt in zip(tokens, tokens[1:]):
                self.counts[current][nxt] += 1

    def next_word_probabilities(self, word: str) -> dict[str, float]:
        followers = self.counts.get(word.lower())
        if not followers:
            raise KeyError(f"'{word}' never appeared in the training corpus")
        total = sum(followers.values())
        return {w: c / total for w, c in followers.most_common()}

    def generate(self, max_words: int = 12, seed: int = 0) -> str:
        if not self.counts:
            raise RuntimeError("train the model before generating text")
        rng = random.Random(seed)
        word, output = START, []
        for _ in range(max_words):
            probs = self.next_word_probabilities(word)
            word = rng.choices(list(probs), weights=list(probs.values()))[0]
            if word == END:
                break
            output.append(word)
        return " ".join(output)


model = BigramModel()
model.train(AGENT_CORPUS)
print("Words the model knows:", len(model.counts))

Words the model knows: 16


### 6.3 Predict the next word

In [33]:
for word in ("agent", "the"):
    print(f"P(next word | '{word}'):")
    for nxt, p in model.next_word_probabilities(word).items():
        print(f"  {nxt:<8} {p:.2f}  {'#' * int(p * 20)}")
    print()

P(next word | 'agent'):
  reads    0.40  ########
  plans    0.20  ####
  calls    0.20  ####
  writes   0.20  ####

P(next word | 'the'):
  agent    0.50  ##########
  task     0.10  ##
  next     0.10  ##
  tool     0.10  ##
  result   0.10  ##
  answer   0.10  ##


"agent" is followed by "reads" in 2 of the 5 sentences that contain "agent", so P = 0.40.

### 6.4 Generate text, and meet hallucination

In [34]:
print(f"seed 7: '{model.generate(seed=7)}'")

seed 7: 'the agent calls a result .'


**"the agent calls a result ."** Every word **pair** in this sentence appeared in the training data ("agent calls", "calls a", "a result"), but the sentence as a whole never did, and it is wrong: agents call *tools*, not results.

This is a tiny **hallucination**. The model produces **likely** text, not **verified** text. LLMs look at much more context, so this happens far less often, but the root cause is the same. That is why AI agents **ground** their answers with tools and retrieval (RAG).

### 🧪 Try it yourself

Generate with different seeds. Which outputs are true, which are fluent nonsense, and which never end?

In [35]:
for seed in range(10):
    print(f"seed {seed}: {model.generate(seed=seed)}")

seed 0: the tool .
seed 1: the result .
seed 2: the answer .
seed 3: the task .
seed 4: the agent reads the agent plans the result .
seed 5: the tool returns a result .
seed 6: the result .
seed 7: the agent calls a result .
seed 8: the answer .
seed 9: the agent reads the agent plans the agent plans the agent reads


---
## 7. Errors are part of the design

Every class validates its input and raises a **clear, specific error** instead of failing mysteriously later. The cell below triggers each error on purpose and catches it, so you can read the messages.

In [36]:
checks = [
    ("search('')", lambda: search("")),
    ("era_for_year(1900)", lambda: era_for_year(1900)),
    ("Eliza().respond('   ')", lambda: Eliza().respond("   ")),
    ("Perceptron(learning_rate=0)", lambda: Perceptron(learning_rate=0)),
    ("ExpertSystem().infer(set())", lambda: ExpertSystem().infer(set())),
    ("model.next_word_probabilities('banana')", lambda: model.next_word_probabilities("banana")),
    ("BigramModel().generate()", lambda: BigramModel().generate()),
]
for label, call in checks:
    try:
        call()
        print(f"{label:<42} -> no error (unexpected!)")
    except (ValueError, KeyError, RuntimeError) as exc:
        print(f"{label:<42} -> {type(exc).__name__}: {exc}")

search('')                                 -> ValueError: keyword must not be empty
era_for_year(1900)                         -> ValueError: No era defined for year 1900; the timeline starts in 1943.
Eliza().respond('   ')                     -> ValueError: text must not be empty
Perceptron(learning_rate=0)                -> ValueError: learning_rate must be positive
ExpertSystem().infer(set())                -> ValueError: at least one fact is required
model.next_word_probabilities('banana')    -> KeyError: "'banana' never appeared in the training corpus"
BigramModel().generate()                   -> RuntimeError: train the model before generating text


---
## 8. What you rebuilt, and what it means for AI agents

| Exhibit | What worked | What failed | What today's agents take from it |
|---|---|---|---|
| ELIZA (1966) | Feels conversational with a few rules | No understanding at all | Beware the ELIZA effect: show sources, admit uncertainty |
| Perceptron (1958) | Learns from examples | Cannot learn XOR (one straight line) | Model capacity matters, so pick a model that fits the task |
| Expert system (1980s) | Exact and explainable | Brittle and expensive to maintain | Use rules for guardrails and compliance, not for everything |
| Backprop (1986) | Hidden layers solve XOR | Needs data, compute and luck with the seed | The training recipe behind every LLM |
| Bigram model | Simple next-word prediction | Fluent nonsense (hallucination) | Ground LLM output with tools and retrieval |

**The big picture:** a modern AI agent is an **LLM** (connectionist, trained with backprop, predicting the next token) placed inside a **perceive → reason → act loop** (an idea from Shakey, 1966–1972), given **tools** and **retrieved knowledge**, and kept safe with **explicit rules** (the expert-system idea). Every exhibit in this notebook lives on inside it.

### Checklist

- [ ] I ran every cell from top to bottom.
- [ ] I can explain why ELIZA sounds smart but is not.
- [ ] I can explain why a perceptron fails on XOR and why a hidden layer fixes it.
- [ ] I can explain forward chaining and why expert systems were brittle.
- [ ] I can explain how next-word prediction leads to hallucination.

### Exercises

1. **(Beginner)** Add an ELIZA rule for `I want (.+)`. Why must it come before the `(.*)` rule?
2. **(Beginner)** Create `NAND_DATA` and train a perceptron on it. Will it succeed? Draw the points first.
3. **(Intermediate)** Write `prove(goal, facts)` for the expert system using **backward chaining**: start from the goal and work back to the facts.
4. **(Intermediate)** Turn `BigramModel` into a **trigram** model that uses the previous two words. Does the generated text improve?
5. **(Advanced)** Build a mini **Shakey**: an agent on a 5×5 grid that perceives walls, plans with A* search (`heapq`) and re-plans when blocked.

**Next lesson:** *Types of AI*, which classifies AI by capability (narrow, general, super) and by functionality.